# Singleton recommendation analysis — history MLP ensemble

This notebook inspects whether the deployed history-MLP ensemble gives sensible answers to the original question: “I like this one book; what else should I read?”

All serving behavior comes from HistoryMLPRecommender. The notebook does not reload checkpoints manually, reproduce the scoring loop, or access model logits directly. It checks:

- qualitative relevance for several contrasting books;
- whether different queries produce different rankings;
- whether recommendations merely repeat the most popular books;
- how many training readers connect each query and recommendation.

In [4]:
from itertools import combinations
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "bookrec").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from bookrec.catalog import normalize_isbn
from bookrec.data import (
    ITEM_COLUMN,
    USER_COLUMN,
    load_dataset,
    split_interactions,
)
from bookrec.implicit import HistoryMLPRecommender

ENSEMBLE_DIRECTORY = ROOT / "artifacts" / "implicit" / "history_mlp_ensemble"
QUERY_TITLES = [
    "The Lord of the Rings",
    "1984",
    "I, Robot",
    "The Hobbit",
    "Harry Potter and the Sorcerer's Stone",
    "ROMEO AND JULIET",
]
TOP_K = 5
MIN_QUERY_INTERACTIONS = 20
MIN_CANDIDATE_INTERACTIONS = 20
INFERENCE_BATCH_SIZE = 16_384
SPLIT_SEED = 42

## 1. Load the production-style recommender

The ensemble, mappings, metadata catalog, training counts, and eligible candidate IDs are loaded once. A future FastAPI application would create this same object during application startup and reuse it for every request.

In [5]:
books = load_dataset("Books.csv")
recommender = HistoryMLPRecommender(
    ENSEMBLE_DIRECTORY,
    books=books,
    min_candidate_interactions=MIN_CANDIDATE_INTERACTIONS,
    inference_batch_size=INFERENCE_BATCH_SIZE,
)
if recommender.loaded.training_item_counts is None:
    raise ValueError("Checkpoint does not contain training item counts")

display(pd.Series({
    "device": str(recommender.device),
    "ensemble members": recommender.member_count,
    "model items": recommender.catalog.model_item_count,
    "items with metadata": recommender.catalog.metadata_item_count,
    "eligible candidates": recommender.candidate_count,
}, name="loaded recommender"))
display(pd.DataFrame({
    "checkpoint": [str(path) for path in recommender.loaded.member_paths]
}))

/home/nuva/Job/Datasentics/.venv/lib/python3.13/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


device                   cuda
ensemble members            5
model items            303422
items with metadata    241431
eligible candidates      5742
Name: loaded recommender, dtype: object

,checkpoint
0,/home/nuva/Job/Datasentics/artifacts/implicit/...
1,/home/nuva/Job/Datasentics/artifacts/implicit/...
2,/home/nuva/Job/Datasentics/artifacts/implicit/...
3,/home/nuva/Job/Datasentics/artifacts/implicit/...
4,/home/nuva/Job/Datasentics/artifacts/implicit/...


## 2. Inspect several singleton queries

Title resolution chooses the most-interacted exact edition meeting the query-support threshold. If no exact edition qualifies, it considers title variants beginning with the requested title. Recommendation inference then receives exactly one resolved ISBN and excludes that ISBN from its candidates.

The returned score is useful for ranking candidates within a query. It is not a calibrated probability that a reader will like the book.

In [6]:
resolved_rows = []
recommendation_frames = []

for requested_title in QUERY_TITLES:
    resolved_isbn = recommender.catalog.resolve_titles(
        [requested_title],
        min_training_interactions=MIN_QUERY_INTERACTIONS,
    )[0]
    resolved = recommender.catalog.get_books([resolved_isbn])[0]
    rankings = recommender.recommend_by_isbn(resolved_isbn, top_k=TOP_K)
    recommendation_books = recommender.catalog.get_books(
        [ranking["isbn"] for ranking in rankings]
    )
    recommendations = pd.DataFrame([
        {**book.to_dict(), "rank": ranking["rank"], "score": ranking["score"]}
        for book, ranking in zip(recommendation_books, rankings)
    ])

    resolved_rows.append({
        "requested title": requested_title,
        "resolved title": resolved.title,
        "query ISBN": resolved.isbn,
        "query training interactions": resolved.training_interactions,
    })
    recommendations.insert(0, "query ISBN", resolved.isbn)
    recommendations.insert(0, "query title", resolved.title)
    recommendation_frames.append(recommendations)

resolved_queries = pd.DataFrame(resolved_rows)
all_recommendations = pd.concat(recommendation_frames, ignore_index=True)

display(resolved_queries)
display(all_recommendations[[
    "query title",
    "rank",
    "title",
    "author",
    "isbn",
    "training_interactions",
    "score",
]])

,requested title,resolved title,query ISBN,query training interactions
0,The Lord of the Rings,The Lord of the Rings (Movie Art Cover),0618129022,58
1,1984,1984,0451524934,163
2,"I, Robot","I, Robot",0553294385,49
3,The Hobbit,The Hobbit : The Enchanting Prelude to The Lor...,0345339681,231
4,Harry Potter and the Sorcerer's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,059035342X,490
5,ROMEO AND JULIET,ROMEO AND JULIET,0671722859,27


,query title,rank,title,author,isbn,training_interactions,score
0,The Lord of the Rings (Movie Art Cover),1,The Mists of Avalon,MARION ZIMMER BRADLEY,0345350499,145,0.985950
1,The Lord of the Rings (Movie Art Cover),2,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,043935806X,278,0.978953
2,The Lord of the Rings (Movie Art Cover),3,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,0590353403,143,0.972657
3,The Lord of the Rings (Movie Art Cover),4,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,0439064864,141,0.971559
4,The Lord of the Rings (Movie Art Cover),5,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,0439139600,157,0.971145
5,1984,1,Brave New World,Aldous Huxley,0060929871,101,0.990794
6,1984,2,Animal Farm,George Orwell,0451526341,132,0.990150
7,1984,3,Slaughterhouse Five or the Children's Crusade:...,Kurt Vonnegut,0440180295,145,0.989927
8,1984,4,The Catcher in the Rye,J.D. Salinger,0316769487,340,0.989317
9,1984,5,Lord of the Flies,William Gerald Golding,0399501487,197,0.989191
